In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

In [2]:
words = open("/Users/raghav/Downloads/names.txt",'r').read().splitlines()

In [3]:
len(words)

32033

In [4]:
words[:5]

['emma', 'olivia', 'ava', 'isabella', 'sophia']

In [6]:
chars = sorted(list(set(''.join(words))))
chars

['a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [10]:
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
stoi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [11]:
itos = {i:s for s,i in stoi.items()}
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [13]:
vocab_size = len(stoi)
vocab_size

27

In [20]:
block_size = 3
def build_dataset(words):
    bs = block_size
    X,Y = [],[]
    for w in words:
        context = [0] * bs
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X,Y

In [21]:
random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xdev,Ydev = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])

Xtr.shape,Ytr.shape

(torch.Size([182437, 3]), torch.Size([182437]))

In [22]:
Xdev.shape, Ydev.shape

(torch.Size([22781, 3]), torch.Size([22781]))

In [53]:
n_embd = 10 
n_hidden = 200
C = torch.randn(vocab_size,n_embd)
W1 = torch.randn((n_embd*block_size),n_hidden)
b1 = torch.randn(n_hidden)
W2 = torch.randn(n_hidden,vocab_size)
b2 = torch.randn(vocab_size)

In [54]:
parameters = [C,W1,b1,W2,b2]
for p in parameters:
    p.requires_grad = True

sum(p.nelement() for p in parameters)

11897

In [55]:
for i in range(200000):
    #mini batch
    ix = torch.randint(0,Xtr.shape[0],(32,))
    #forward pass
    emb = C[Xtr[ix]]
    emb = emb.view(-1,(n_embd*block_size))
    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    #loss
    loss = F.cross_entropy(logits,Ytr[ix])
    if i % 10000 == 0:
        print(loss.item())
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    if i < 100000:
        lr = 0.1
    else:
        lr = 0.01
    for p in parameters:
        p.data += -lr * p.grad



28.50118064880371
3.032623052597046
2.4790263175964355
2.305713653564453
1.98075270652771
2.990861415863037
3.086543560028076
2.5071282386779785
2.4890031814575195
2.2062439918518066
1.8681145906448364
2.112138032913208
2.081993579864502
1.7620153427124023
2.0248708724975586
2.3146588802337646
2.183445930480957
1.8812607526779175
2.2130348682403564
2.0876405239105225


In [56]:
with torch.no_grad():
    emb = C[Xtr]
    h = torch.tanh(emb.view(-1,(n_embd*block_size) ) @ W1 + b1)
    logits = h @ W2 + b2
    train_loss = F.cross_entropy(logits, Ytr)
    print("train:", train_loss.item())

train: 2.1191718578338623


In [58]:
with torch.no_grad():
    emb = C[Xdev]
    h = torch.tanh(emb.view(-1, (n_embd*block_size)) @ W1 + b1)
    logits = h @ W2 + b2
    train_loss = F.cross_entropy(logits, Ydev)
    print("dev:", train_loss.item())

train: 2.1826372146606445


In [61]:
n_embd = 10 
n_hidden = 200
C = torch.randn(vocab_size,n_embd)
W1 = torch.randn((n_embd*block_size),n_hidden)
b1 = torch.randn(n_hidden)
W2 = torch.randn(n_hidden,vocab_size) * 0.01
b2 = torch.randn(vocab_size) * 0 

In [62]:
parameters = [C,W1,b1,W2,b2]
for p in parameters:
    p.requires_grad = True

sum(p.nelement() for p in parameters)

11897

In [63]:
for i in range(200000):
    #mini batch
    ix = torch.randint(0,Xtr.shape[0],(32,))
    #forward pass
    emb = C[Xtr[ix]]
    emb = emb.view(-1,(n_embd*block_size))
    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    #loss
    loss = F.cross_entropy(logits,Ytr[ix])
    if i % 10000 == 0:
        print(loss.item())
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    if i < 100000:
        lr = 0.1
    else:
        lr = 0.01
    for p in parameters:
        p.data += -lr * p.grad



3.2755463123321533
2.5509486198425293
2.724820852279663
2.0408096313476562
2.506202459335327
2.0886664390563965
2.483126401901245
2.4849298000335693
2.1023476123809814
2.0406432151794434
1.9559085369110107
1.8144714832305908
1.9896681308746338
1.856499433517456
2.1665024757385254
2.0059635639190674
2.1915395259857178
2.1375317573547363
2.2890448570251465
2.2987351417541504


In [64]:
with torch.no_grad():
    emb = C[Xtr]
    h = torch.tanh(emb.view(-1,(n_embd*block_size) ) @ W1 + b1)
    logits = h @ W2 + b2
    train_loss = F.cross_entropy(logits, Ytr)
    print("train:", train_loss.item())

train: 2.067728281021118


In [65]:
with torch.no_grad():
    emb = C[Xdev]
    h = torch.tanh(emb.view(-1, (n_embd*block_size)) @ W1 + b1)
    logits = h @ W2 + b2
    train_loss = F.cross_entropy(logits, Ydev)
    print("dev:", train_loss.item())

dev: 2.151384115219116


In [66]:
with torch.no_grad():
    emb = C[Xtr[:1000]]
    embcat = emb.view(-1, 30)
    h_pre = embcat @ W1 + b1
    h = torch.tanh(h_pre)
    print(h.abs().max().item())
    print((h.abs() > 0.97).float().mean().item())

1.0
0.7156999707221985


In [73]:
n_embd = 10 
n_hidden = 200
C = torch.randn(vocab_size,n_embd)
W1 = (5/3)*torch.randn((n_embd*block_size),n_hidden) / (n_embd*block_size) ** 0.5 
b1 = torch.zeros(n_hidden)
W2 = torch.randn(n_hidden,vocab_size) * 0.01
b2 = torch.zeros(vocab_size)

In [74]:
parameters = [C,W1,b1,W2,b2]
for p in parameters:
    p.requires_grad = True

sum(p.nelement() for p in parameters)

11897

In [75]:
for i in range(200000):
    #mini batch
    ix = torch.randint(0,Xtr.shape[0],(32,))
    #forward pass
    emb = C[Xtr[ix]]
    emb = emb.view(-1,(n_embd*block_size))
    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    #loss
    loss = F.cross_entropy(logits,Ytr[ix])
    if i % 10000 == 0:
        print(loss.item())
    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()

    #update
    if i < 100000:
        lr = 0.1
    else:
        lr = 0.01
    for p in parameters:
        p.data += -lr * p.grad



3.3238189220428467
1.9239386320114136
2.5059332847595215
2.2896127700805664
2.246016025543213
1.6717679500579834
2.436998128890991
1.4549529552459717
2.094550848007202
2.313446283340454
1.952379822731018
2.1464028358459473
1.9458668231964111
2.046738862991333
1.991505742073059
1.818562388420105
1.9162487983703613
1.820998191833496
2.5117931365966797
2.0985071659088135


In [76]:
with torch.no_grad():
    emb = C[Xtr]
    h = torch.tanh(emb.view(-1,(n_embd*block_size) ) @ W1 + b1)
    logits = h @ W2 + b2
    train_loss = F.cross_entropy(logits, Ytr)
    print("train:", train_loss.item())

train: 2.037367582321167


In [77]:
with torch.no_grad():
    emb = C[Xdev]
    h = torch.tanh(emb.view(-1, (n_embd*block_size)) @ W1 + b1)
    logits = h @ W2 + b2
    train_loss = F.cross_entropy(logits, Ydev)
    print("dev:", train_loss.item())

dev: 2.123256206512451


In [78]:
with torch.no_grad():
    emb = C[Xtr[:1000]]
    embcat = emb.view(-1, 30)
    h_pre = embcat @ W1 + b1
    h = torch.tanh(h_pre)
    print(h.abs().max().item())
    print((h.abs() > 0.97).float().mean().item())

1.0
0.3728350102901459


In [79]:
n_embd = 10 
n_hidden = 200
C = torch.randn(vocab_size,n_embd)
W1 = (5/3)*torch.randn((n_embd*block_size),n_hidden) / (n_embd*block_size) ** 0.5 
b1 = torch.zeros(n_hidden)
W2 = torch.randn(n_hidden,vocab_size) * 0.01
b2 = torch.zeros(vocab_size)
bngain = torch.ones(1,n_hidden) 
bnbias = torch.zeros(1,n_hidden)

In [80]:
parameters = [C,W1,b1,W2,b2,bngain,bnbias]
for p in parameters:
    p.requires_grad = True

sum(p.nelement() for p in parameters)

12297

In [90]:
bnmean_running = torch.zeros((1, n_hidden))
bnstd_running  = torch.ones((1, n_hidden))

for i in range(200000):
    #mini batch
    ix = torch.randint(0,Xtr.shape[0], (32,))

    emb = C[Xtr[ix]]
    embcat = emb.view(-1,(n_embd*block_size))
    h_pre = embcat @ W1 + b1
    bnmean = h_pre.mean(0,keepdim = True)
    bnstd = h_pre.std(0, keepdim = True)
    with torch.no_grad():
        bnmean_running = 0.999 * bnmean_running + 0.001 * bnmean
        bnstd_running  = 0.999 * bnstd_running + 0.001 * bnstd
    h_hat = (h_pre - bnmean) / (bnstd + 1e-5)
    h_bn = bngain * h_hat + bnbias
    h = torch.tanh(h_bn)
    logits = h @ W2 + b2

    loss = F.cross_entropy(logits,Ytr[ix])
    if i%10000 == 0:
        print(loss.item())

    for p in parameters:
        p.grad = None 

    loss.backward()

    if i < 100000:
        lr = 0.1
    else:
        lr = 0.01

    for p in parameters:
        p.data += -0.1 * p.grad

2.2526333332061768
1.9454940557479858
2.183521270751953
2.474355459213257
2.5984976291656494
2.110884666442871
2.5223660469055176
2.649482250213623
1.8965412378311157
2.2939183712005615
2.273001194000244
2.007195472717285
2.5224809646606445
2.6760637760162354
2.9840264320373535
2.085777521133423
1.9803372621536255
2.700695753097534
2.41445255279541
1.902424931526184


In [91]:
with torch.no_grad():
    emb = C[Xtr]
    embcat = emb.view(-1, (n_embd*block_size))
    h_pre = embcat @ W1 + b1
    h_hat = (h_pre - bnmean_running) / (bnstd_running + 1e-5)
    h_bn  = bngain * h_hat + bnbias
    h = torch.tanh(h_bn)
    logits = h @ W2 + b2
    train_loss = F.cross_entropy(logits, Ytr)
    print("train:", train_loss.item())

train: 2.113037586212158


In [92]:
with torch.no_grad():
    emb = C[Xdev]
    embcat = emb.view(-1, (n_embd*block_size))
    h_pre = embcat @ W1 + b1
    h_hat = (h_pre - bnmean_running) / (bnstd_running + 1e-5)
    h_bn  = bngain * h_hat + bnbias
    h = torch.tanh(h_bn)
    logits = h @ W2 + b2
    train_loss = F.cross_entropy(logits, Ydev)
    print("dev:", train_loss.item())

train: 2.183140993118286


In [85]:
import torch
true_mean = 3.0
running = 0.0

for i in range(10000):
    batch_mean = true_mean + torch.randn(1).item() * 0.1  # noisy estimate
    running = 0.999 * running + 0.001 * batch_mean

print(running)  # should be close to 3.0

3.001337928563554


In [94]:
class Linear:

    def __init__(self,fan_in,fan_out,bias=True):
        self.weight = torch.randn((fan_in,fan_out),generator = g) / fan_in ** 0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self,x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
            
        return self.out

    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])

class BatchNorm1d:

    def __init__(self,dim,eps = 1e-5,momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        #parameters
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        #buffers
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self,x):

        if self.training:
            xmean = x.mean(0,keepdim = True)
            xvar = x.var(0,keepdim = True)
        else:
            xmean = self.running_mean
            xvar = self.running_var

        xhat = (x-xmean) / torch.sqrt(xvar+self.eps)
        self.out = self.gamma * xhat + self.beta

        if self.training:
            with torch.no_grad():
                self.running_mean = (1- self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar

        return self.out

    def parameters (self):
        return [self.gamma,self.beta]


class Tanh():
    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self):
        return [] 
    

In [110]:
g = torch.Generator().manual_seed(2147483647)

n_emd = 10
n_hidden = 100

layers = [
    Linear((n_emd*block_size),100,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,100,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,100,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,100,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,100,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden, vocab_size, bias=False), BatchNorm1d(vocab_size),
]

In [104]:
C = torch.randn((vocab_size, n_embd), generator=g)
parameters = [C] + [p for layer in layers for p in layer.parameters()]
print(sum(p.nelement() for p in parameters))

47024


22781